# 02 · Multimodal RAG with an open-source LLM + evaluation (Kaggle GPU)

Compares three systems on the MMLongBench-Doc subset:

| mode | what it is |
|---|---|
| `no_rag` | the LLM alone (hallucination baseline) |
| `text_only` | classic RAG over plain text only |
| `multimodal` | **our system** — text + tables + figures (CLIP) + the VLM sees retrieved images |

LLM: **Qwen2.5-VL-3B-Instruct** (vision-language, fp16 on a T4).

**Before running:** GPU on, Internet on, and *Add Input* → the output of notebook 01
(or upload your `artifacts.zip` as a Kaggle dataset).

In [ ]:
# repository and branch to run
REPO_URL = "https://github.com/yayyyyshi/multimodal_rag.git"
BRANCH   = "main"  # use "feature/full-pipeline" until the PR is merged
import os, subprocess
os.chdir("/kaggle/working")
if not os.path.exists("multimodal_rag"):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL], check=True)
os.chdir("/kaggle/working/multimodal_rag")
!git log --oneline -3

In [ ]:
!pip install -q pymupdf chromadb sentence-transformers pyyaml 2>&1 | tail -2
import glob, zipfile, os
cands = glob.glob("/kaggle/input/**/artifacts.zip", recursive=True)
print(cands)
assert cands, "Add notebook 01's output (or a dataset containing artifacts.zip) as input"
zipfile.ZipFile(cands[0]).extractall(".")
!ls artifacts | head

In [ ]:
# data/eval/questions.jsonl is in the repo, so only the index is needed
!python scripts/build_index.py

## Try a question

In [ ]:
import sys; sys.path.insert(0, ".")
from src.pipeline.rag import MultimodalRAG
rag = MultimodalRAG.from_config(backend="transformers_vl")
out = rag.answer("According to the report, how do 5% of the Latinos see economic upward mobility for their children?")
print(out["answer"]); print(out["citations"])

## Full evaluation (about 30 to 60 min). Use `--limit 40` for a quick run.

In [ ]:
!python scripts/evaluate.py --modes no_rag text_only multimodal --backend transformers_vl --run-name kaggle_full

In [ ]:
import json, matplotlib.pyplot as plt
S = json.load(open("results/kaggle_full/summary.json"))["summary"]
modes = list(S)
metrics = [("accuracy", "Accuracy ↑"), ("hallucination_rate", "Hallucination rate ↓"), ("ret_recall@5", "Retrieval Recall@5 ↑")]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, (k, title) in zip(axes, metrics):
    vals = [S[m].get(k) or 0 for m in modes]
    ax.bar(modes, vals, color=["#9aa5b1", "#5b8def", "#1f4e9c"][:len(modes)])
    ax.set_title(title); ax.set_ylim(0, 1)
    for i, v in enumerate(vals): ax.text(i, v + 0.02, f"{v:.2f}", ha="center")
plt.tight_layout(); plt.savefig("results/kaggle_full/comparison.png", dpi=150); plt.show()
print(open("results/kaggle_full/summary.md").read())

In [ ]:
!cd results && zip -qr ../results_kaggle_full.zip kaggle_full && ls -la ../results_kaggle_full.zip